# DTI Prediction — KIBA Dataset (v2)

Architecture identical to the proven DrugBank v17 notebook.
Handles d_2D / d_3D / t_feat length mismatch by keeping only the
rows where ALL four arrays share the same valid index.

In [1]:
# ================================================================
# Cell 1: Imports & GPU check
# ================================================================
import os, gc, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, matthews_corrcoef,
    precision_score, recall_score, confusion_matrix
)
warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# ================================================================
# Cell 2: Load raw data & reconcile length mismatch
#
# KIBA reality: d_2D has 116 339 rows, the other three have 116 350.
# Strategy: the shortest array (d_2D) is the bottleneck.
# We keep only the first N_VALID rows where N_VALID = min of all lengths.
# This is safe because EviDTI builds all arrays with the same row order
# from the CSV; any length difference is due to a few trailing rows that
# failed feature extraction in d_2D.  Dropping them is correct.
# ================================================================
DATA_PATH = '/kaggle/input/datasets/shiyunsg/kiba-dataset'

d_2D_raw   = np.load(f'{DATA_PATH}/KIBA_d_2d_feature.npy',  allow_pickle=True)
d_3D_raw   = np.load(f'{DATA_PATH}/KIBA_d_3d_feature.npy',  allow_pickle=True)
t_feat_raw = np.load(f'{DATA_PATH}/KIBA_t_1D_feature.npy',  allow_pickle=True)
df_full    = pd.read_csv(f'{DATA_PATH}/KIBA_total_cid_unid.csv')

print(f'd_2D   : {d_2D_raw.shape}, {d_2D_raw.dtype}')
print(f'd_3D   : {d_3D_raw.shape}, {d_3D_raw.dtype}')
print(f't_feat : {t_feat_raw.shape}, {t_feat_raw.dtype}')
print(f'df     : {df_full.shape}')
print(f'df cols: {df_full.columns.tolist()}')

# ── Reconcile: take the intersection length ──────────────────────
N_VALID = min(len(d_2D_raw), len(d_3D_raw), len(t_feat_raw), len(df_full))
print(f'\nArray lengths: d_2D={len(d_2D_raw)}, d_3D={len(d_3D_raw)}, '
      f't_feat={len(t_feat_raw)}, df={len(df_full)}')
print(f'Using first N_VALID={N_VALID} rows (dropping {max(len(d_2D_raw),len(d_3D_raw),len(t_feat_raw),len(df_full))-N_VALID} tail rows).')

d_2D_raw   = d_2D_raw[:N_VALID]
d_3D_raw   = d_3D_raw[:N_VALID]
t_feat_raw = t_feat_raw[:N_VALID]
df         = df_full.iloc[:N_VALID].reset_index(drop=True)

assert len(d_2D_raw) == len(d_3D_raw) == len(t_feat_raw) == len(df), 'Still mismatched!'
print('Row alignment OK.')

# ── Label column detection ────────────────────────────────────────
label_col = None
for c in df.columns:
    if c.lower() in ('label', 'interaction', 'y', 'activity', 'active'):
        label_col = c
        break
if label_col is None:
    label_col = df.columns[-1]
print(f'Label column: "{label_col}"')

labels_all = df[label_col].values.astype(int)
D2_DIM     = d_2D_raw.shape[1]
print(f'D2_DIM={D2_DIM}, Pos={(labels_all==1).sum()}, Neg={(labels_all==0).sum()}')

d_2D   : (116339, 256), float32
d_3D   : (116350,), object
t_feat : (116350,), object
df     : (116350, 5)
df cols: ['cid', 'uid', 'smiles', 'seq', 'label']

Array lengths: d_2D=116339, d_3D=116350, t_feat=116350, df=116350
Using first N_VALID=116339 rows (dropping 11 tail rows).
Row alignment OK.
Label column: "label"
D2_DIM=256, Pos=22153, Neg=94186


In [3]:
# ================================================================
# Cell 3: Build d_3D feature array (515-dim)
# ================================================================
ATOM_SCALAR_KEYS  = ['atomic_num','chiral_tag','degree','explicit_valence',
                     'formal_charge','hybridization','implicit_valence',
                     'is_aromatic','total_numHs','mass']
ATOM_VEC_KEYS     = ['atom_pos']           # (n_atoms,3) → mean → 3
BOND_SCALAR_KEYS  = ['bond_dir','bond_type','is_in_ring','bond_length']
ANGLE_SCALAR_KEYS = ['bond_angle','Ba_bond_angle','Bl_bond_length']
AD_DIST_KEY       = 'Ad_atom_dist'
FIXED_FP_KEYS     = ['morgan_fp','maccs_fp','daylight_fg_counts']
FP_DIMS           = [200, 167, 127]

D3_DIM = (len(ATOM_SCALAR_KEYS) + 3 + len(BOND_SCALAR_KEYS) +
          len(ANGLE_SCALAR_KEYS) + 1 + sum(FP_DIMS))  # = 515
print(f'D3_DIM = {D3_DIM}')


def safe_mean(arr):
    return float(arr.mean()) if len(arr) > 0 else 0.0

def safe_mean_vec(arr, ndim):
    return arr.mean(axis=0).flatten().astype(np.float32) if arr.shape[0] > 0 else np.zeros(ndim, np.float32)

def dict_to_vec(d):
    parts = []
    # Atom scalars
    for k in ATOM_SCALAR_KEYS:
        v = d.get(k)
        arr = np.asarray(v, np.float32).flatten() if v is not None else np.array([], np.float32)
        parts.append(np.array([safe_mean(arr)], np.float32))
    # atom_pos → 3 dims
    v = d.get('atom_pos')
    if v is None:
        parts.append(np.zeros(3, np.float32))
    else:
        arr = np.asarray(v, np.float32)
        if arr.ndim == 1: arr = arr.reshape(-1, 1)
        parts.append(safe_mean_vec(arr, arr.shape[1]))
    # Bond scalars
    for k in BOND_SCALAR_KEYS:
        v = d.get(k)
        arr = np.asarray(v, np.float32).flatten() if v is not None else np.array([], np.float32)
        parts.append(np.array([safe_mean(arr)], np.float32))
    # Angle scalars
    for k in ANGLE_SCALAR_KEYS:
        v = d.get(k)
        arr = np.asarray(v, np.float32).flatten() if v is not None else np.array([], np.float32)
        parts.append(np.array([safe_mean(arr)], np.float32))
    # Ad_atom_dist
    v = d.get(AD_DIST_KEY)
    arr = np.asarray(v, np.float32).flatten() if v is not None else np.array([], np.float32)
    parts.append(np.array([safe_mean(arr)], np.float32))
    # Fixed fingerprints
    for k, dim in zip(FIXED_FP_KEYS, FP_DIMS):
        v = d.get(k)
        if v is None:
            parts.append(np.zeros(dim, np.float32))
        else:
            arr = np.asarray(v, np.float32).flatten()
            if len(arr) != dim:
                tmp = np.zeros(dim, np.float32)
                tmp[:min(len(arr), dim)] = arr[:dim]
                arr = tmp
            parts.append(np.nan_to_num(arr, nan=0., posinf=0., neginf=0.))

    result = np.nan_to_num(np.concatenate(parts).astype(np.float32),
                           nan=0., posinf=0., neginf=0.)
    assert len(result) == D3_DIM
    return result


# Validate
tv = dict_to_vec(d_3D_raw[0])
print(f'Test vec[0]: shape={tv.shape}, NaNs={np.isnan(tv).sum()}')

print(f'Building d_3D_arr ({N_VALID} rows)...')
d_3D_arr = np.zeros((N_VALID, D3_DIM), np.float32)
for i in range(N_VALID):
    d_3D_arr[i] = dict_to_vec(d_3D_raw[i])

print(f'd_3D_arr: {d_3D_arr.shape}')
assert np.isnan(d_3D_arr).sum() == 0 and np.isinf(d_3D_arr).sum() == 0
print('d_3D_arr OK.')
del d_3D_raw; gc.collect()

D3_DIM = 515
Test vec[0]: shape=(515,), NaNs=0
Building d_3D_arr (116339 rows)...
d_3D_arr: (116339, 515)
d_3D_arr OK.


0

In [4]:
# ================================================================
# Cell 4: Balanced sampling  (≤15 000 per class → ≤30 000 total)
# Safe for T4 RAM <29 GB.
# ================================================================
MAX_PER_CLASS = 15000
SEED = 42
np.random.seed(SEED)

pos_idx = np.where(labels_all == 1)[0]
neg_idx = np.where(labels_all == 0)[0]
print(f'Full dataset: pos={len(pos_idx)}, neg={len(neg_idx)}')

sampled_pos = np.random.choice(pos_idx, min(MAX_PER_CLASS, len(pos_idx)), replace=False)
sampled_neg = np.random.choice(neg_idx, min(MAX_PER_CLASS, len(neg_idx)), replace=False)
sampled_idx = np.concatenate([sampled_pos, sampled_neg])
np.random.shuffle(sampled_idx)
sampled_labels = labels_all[sampled_idx]

print(f'Sampled: total={len(sampled_idx)}, '
      f'pos={(sampled_labels==1).sum()}, neg={(sampled_labels==0).sum()}')

Full dataset: pos=22153, neg=94186
Sampled: total=30000, pos=15000, neg=15000


In [5]:
# ================================================================
# Cell 5: Dataset & DataLoader
# ================================================================
MAX_PROT_LEN = 800
P_EMB_DIM    = 1024
BATCH_SIZE   = 128
print(f'Dims: D2={D2_DIM}, D3={D3_DIM}, P={P_EMB_DIM}, MAX_PROT_LEN={MAX_PROT_LEN}')


class DTIDataset(Dataset):
    def __init__(self, pair_indices, labels, d_2D_raw, d_3D_arr, t_feat_raw,
                 max_prot_len=MAX_PROT_LEN):
        self.pair_indices = pair_indices
        self.labels       = labels
        self.d_2D         = d_2D_raw
        self.d_3D         = d_3D_arr
        self.t_feat       = t_feat_raw
        self.max_prot_len = max_prot_len

    def __len__(self):
        return len(self.pair_indices)

    def __getitem__(self, idx):
        i   = int(self.pair_indices[idx])
        lbl = int(self.labels[idx])
        d2  = self.d_2D[i].astype(np.float32)
        d3  = self.d_3D[i].astype(np.float32)
        p   = self.t_feat[i]
        if not isinstance(p, np.ndarray):
            p = np.array(p, dtype=np.float32)
        else:
            p = p.astype(np.float32)
        if p.ndim == 1:
            p = p.reshape(1, -1)
        p = p[:self.max_prot_len]
        return (torch.from_numpy(d2), torch.from_numpy(d3),
                torch.from_numpy(p), lbl)


def collate_fn(batch):
    d2_list, d3_list, p_list, lbl_list = zip(*batch)
    d2   = torch.stack(d2_list, 0)
    d3   = torch.stack(d3_list, 0)
    lbls = torch.tensor(lbl_list, dtype=torch.long)
    lens    = [p.shape[0] for p in p_list]
    max_len = max(lens)
    P_DIM   = p_list[0].shape[1]
    B       = len(p_list)
    p_pad   = torch.zeros(B, max_len, P_DIM)
    mask    = torch.zeros(B, max_len, dtype=torch.bool)
    for i, (p, L) in enumerate(zip(p_list, lens)):
        p_pad[i, :L] = p
        mask[i, :L]  = True
    return d2, d3, p_pad, mask, lbls


local_idx = np.arange(len(sampled_idx))
tr_loc, tmp_loc, _, tmp_lbl = train_test_split(
    local_idx, sampled_labels, test_size=0.20,
    stratify=sampled_labels, random_state=1)
val_loc, test_loc = train_test_split(
    tmp_loc, test_size=0.50,
    stratify=sampled_labels[tmp_loc], random_state=1)

for name, loc in [('Train', tr_loc), ('Val', val_loc), ('Test', test_loc)]:
    pos_n = sampled_labels[loc].sum()
    print(f'{name:5s}: {len(loc):6d}  pos={pos_n} ({100*pos_n/len(loc):.1f}%)')


def make_loader(loc, shuffle):
    ds = DTIDataset(
        pair_indices=sampled_idx[loc], labels=sampled_labels[loc],
        d_2D_raw=d_2D_raw, d_3D_arr=d_3D_arr, t_feat_raw=t_feat_raw)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=collate_fn, num_workers=2, pin_memory=True)


train_loader = make_loader(tr_loc,   shuffle=True)
val_loader   = make_loader(val_loc,  shuffle=False)
test_loader  = make_loader(test_loc, shuffle=False)
print(f'Batches — Train:{len(train_loader)} Val:{len(val_loader)} Test:{len(test_loader)}')

Dims: D2=256, D3=515, P=1024, MAX_PROT_LEN=800
Train:  24000  pos=12000 (50.0%)
Val  :   3000  pos=1500 (50.0%)
Test :   3000  pos=1500 (50.0%)
Batches — Train:188 Val:24 Test:24


In [6]:
# ================================================================
# Cell 6: Model (identical to DrugBank v17)
# ================================================================

class LightAttentionPooling(nn.Module):
    def __init__(self, in_dim, out_dim, kernel=9, dropout=0.2):
        super().__init__()
        pad = kernel // 2
        self.feat_conv = nn.Conv1d(in_dim, in_dim, kernel, padding=pad)
        self.attn_conv = nn.Conv1d(in_dim, in_dim, kernel, padding=pad)
        self.proj = nn.Sequential(
            nn.Linear(2 * in_dim, out_dim),
            nn.LayerNorm(out_dim), nn.Dropout(dropout), nn.GELU()
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        x      = x.float()
        xt     = x.permute(0, 2, 1)             # [B, C, L]
        feat   = self.drop(self.feat_conv(xt))  # [B, C, L]
        attn   = self.attn_conv(xt)             # [B, C, L]
        mask_t = mask.unsqueeze(1)              # [B, 1, L]
        attn   = attn.masked_fill(~mask_t, -1e4)
        attn   = torch.softmax(attn, dim=2)
        weighted = (feat * attn).sum(dim=2)
        maxpool  = feat.masked_fill(~mask_t, -1e4).max(dim=2).values
        return self.proj(torch.cat([weighted, maxpool], dim=1))


class DrugEncoder(nn.Module):
    def __init__(self, d2_dim, d3_dim, out_dim, dropout=0.2):
        super().__init__()
        self.d2_enc = nn.Sequential(
            nn.Linear(d2_dim, 512), nn.LayerNorm(512), nn.Dropout(dropout), nn.GELU(),
            nn.Linear(512, out_dim), nn.LayerNorm(out_dim), nn.Dropout(dropout), nn.GELU()
        )
        self.d3_enc = nn.Sequential(
            nn.Linear(d3_dim, 512), nn.LayerNorm(512), nn.Dropout(dropout), nn.GELU(),
            nn.Linear(512, out_dim), nn.LayerNorm(out_dim), nn.Dropout(dropout), nn.GELU()
        )
        self.fusion = nn.Sequential(
            nn.Linear(out_dim * 2, out_dim),
            nn.LayerNorm(out_dim), nn.Dropout(dropout), nn.GELU()
        )

    def forward(self, d2, d3):
        return self.fusion(torch.cat([self.d2_enc(d2), self.d3_enc(d3)], dim=1))


class DTIModel(nn.Module):
    def __init__(self, d2_dim, d3_dim, p_dim=1024,
                 drug_out=256, prot_out=256, hidden=512, dropout=0.2):
        super().__init__()
        self.drug_enc  = DrugEncoder(d2_dim, d3_dim, drug_out, dropout)
        self.prot_enc  = LightAttentionPooling(p_dim, prot_out, kernel=9, dropout=dropout)
        ca_dim = 256
        self.drug_proj = nn.Linear(drug_out, ca_dim)
        self.prot_proj = nn.Linear(prot_out, ca_dim)
        self.ca_drug   = nn.MultiheadAttention(ca_dim, num_heads=4, dropout=dropout, batch_first=True)
        self.ca_prot   = nn.MultiheadAttention(ca_dim, num_heads=4, dropout=dropout, batch_first=True)
        self.ca_norm_d = nn.LayerNorm(ca_dim)
        self.ca_norm_p = nn.LayerNorm(ca_dim)
        self.decoder = nn.Sequential(
            nn.Linear(ca_dim * 2, hidden),
            nn.LayerNorm(hidden), nn.Dropout(dropout), nn.GELU(),
            nn.Linear(hidden, hidden // 2),
            nn.LayerNorm(hidden // 2), nn.Dropout(dropout), nn.GELU(),
            nn.Linear(hidden // 2, 2)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, d2, d3, p, mask):
        drug_h = self.drug_enc(d2, d3)
        prot_h = self.prot_enc(p, mask)
        dq = self.drug_proj(drug_h).unsqueeze(1)
        pk = self.prot_proj(prot_h).unsqueeze(1)
        d2p, _ = self.ca_drug(dq, pk, pk)
        p2d, _ = self.ca_prot(pk, dq, dq)
        drug_ca = self.ca_norm_d(self.drug_proj(drug_h) + d2p.squeeze(1))
        prot_ca = self.ca_norm_p(self.prot_proj(prot_h) + p2d.squeeze(1))
        return self.decoder(torch.cat([drug_ca, prot_ca], dim=1))


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = DTIModel(
    d2_dim=D2_DIM, d3_dim=D3_DIM, p_dim=P_EMB_DIM,
    drug_out=256, prot_out=256, hidden=512, dropout=0.2
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model params: {n_params:,}')

# ── Sanity check ─────────────────────────────────────────────────
d2b, d3b, pb, mkb, lblb = next(iter(train_loader))
assert not d2b.isnan().any(), 'NaN in d2!'
assert not d3b.isnan().any(), 'NaN in d3!'
assert not pb.isnan().any(),  'NaN in protein!'
print(f'd2 range: [{d2b.min():.3f}, {d2b.max():.3f}]')
print(f'd3 range: [{d3b.min():.3f}, {d3b.max():.3f}]')
print(f'p  range: [{pb.min():.3f}, {pb.max():.3f}]')
with torch.no_grad():
    out = model(d2b.to(device).float(), d3b.to(device).float(),
                pb.to(device).float(), mkb.to(device))
assert not out.isnan().any(), 'NaN in model output!'
prob = torch.softmax(out.float(), dim=1)[:, 1]
print(f'Output shape: {out.shape},  prob mean={prob.mean():.3f}, std={prob.std():.4f}')
assert prob.std() > 0.005, 'COLLAPSED output!'
print('Sanity check PASSED.')

Model params: 21,249,794
d2 range: [-3.902, 3.472]
d3 range: [-1.671, 41.000]
p  range: [-1.610, 1.632]
Output shape: torch.Size([128, 2]),  prob mean=0.729, std=0.1886
Sanity check PASSED.


In [7]:
# ================================================================
# Cell 7: Training
# OneCycleLR stepped once per batch after optimizer.step().
# ================================================================
MAX_EPOCH = 30
BASE_LR   = 3e-4
WD        = 1e-4
PATIENCE  = 20
CKPT      = '/kaggle/working/best_model_kiba.pt'

optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr           = BASE_LR,
    total_steps      = MAX_EPOCH * len(train_loader),
    pct_start        = 0.10,
    anneal_strategy  = 'cos',
    div_factor       = 25.0,
    final_div_factor = 1000.0,
)
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None
print(f'LR={BASE_LR}, BS={BATCH_SIZE}, MaxEpoch={MAX_EPOCH}')


def run_epoch(loader, ep, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for step, (d2, d3, p, mask, lbl) in enumerate(loader):
            d2   = d2.to(device, non_blocking=True).float()
            d3   = d3.to(device, non_blocking=True).float()
            p    = p.to(device, non_blocking=True).float()
            mask = mask.to(device, non_blocking=True)
            lbl  = lbl.to(device, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            if scaler and train:
                with torch.cuda.amp.autocast():
                    logits = model(d2, d3, p, mask)
                    loss   = F.cross_entropy(logits, lbl, label_smoothing=0.05)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()  # ← after optimizer.step(), once per batch
            else:
                logits = model(d2, d3, p, mask)
                loss   = F.cross_entropy(logits, lbl, label_smoothing=0.05)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()

            total_loss += loss.item()
            with torch.no_grad():
                prob = torch.softmax(logits.float(), dim=1)[:, 1]
                pred = logits.argmax(dim=1)
            all_preds.append(pred.cpu())
            all_labels.append(lbl.cpu())
            all_probs.append(prob.cpu())

            if train and step % 50 == 49:
                ba = (pred.cpu() == lbl.cpu()).float().mean().item() * 100
                print(f'  Ep{ep} [{step+1:3d}/{len(loader)}] '
                      f'loss={loss.item():.4f} acc={ba:.1f}% '
                      f'std={prob.cpu().std():.4f} '
                      f'LR={optimizer.param_groups[0]["lr"]:.2e}')

    preds    = torch.cat(all_preds).numpy()
    labels   = torch.cat(all_labels).numpy()
    probs    = torch.cat(all_probs).numpy()
    avg_loss = total_loss / max(len(loader), 1)
    acc      = float((preds == labels).mean())
    return avg_loss, acc, preds, labels, probs


best_auc   = 0.0
no_improve = 0
start_time = time.time()

for ep in range(1, MAX_EPOCH + 1):
    tr_loss, tr_acc, _, _, _ = run_epoch(train_loader, ep, train=True)

    val_loss, val_acc, val_preds, val_labels, val_probs = run_epoch(
        val_loader, ep, train=False)

    try:
        val_auc  = roc_auc_score(val_labels, val_probs)
        val_aupr = average_precision_score(val_labels, val_probs)
    except Exception:
        val_auc = val_aupr = float('nan')

    val_f1   = f1_score(val_labels, val_preds, zero_division=0)
    val_mcc  = matthews_corrcoef(val_labels, val_preds)
    val_prec = precision_score(val_labels, val_preds, zero_division=0)
    val_rec  = recall_score(val_labels, val_preds, zero_division=0)
    elapsed  = (time.time() - start_time) / 60

    print(f'[Ep {ep:3d}/{MAX_EPOCH}] '
          f'tr={100*tr_acc:.1f}% val={100*val_acc:.1f}% '
          f'AUC={val_auc:.3f} AUPR={val_aupr:.3f} '
          f'F1={val_f1:.3f} MCC={val_mcc:.3f} '
          f'Prec={val_prec:.3f} Rec={val_rec:.3f} '
          f'({elapsed:.1f}min)')

    is_nan = val_auc != val_auc
    if (not is_nan) and val_auc > best_auc:
        best_auc   = val_auc
        no_improve = 0
        torch.save(model.state_dict(), CKPT)
        print(f'  ★ Best AUC={best_auc:.4f} saved.')
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f'Early stop at epoch {ep}.')
        break

print(f'Done. Best val AUC={best_auc:.4f}')

LR=0.0003, BS=128, MaxEpoch=30
  Ep1 [ 50/188] loss=0.7364 acc=55.5% std=0.1895 LR=1.76e-05
  Ep1 [100/188] loss=0.6697 acc=58.6% std=0.1662 LR=3.38e-05
  Ep1 [150/188] loss=0.7153 acc=57.8% std=0.1805 LR=5.96e-05
[Ep   1/30] tr=57.0% val=67.2% AUC=0.736 AUPR=0.732 F1=0.689 MCC=0.346 Prec=0.655 Rec=0.726 (2.3min)
  ★ Best AUC=0.7358 saved.
  Ep2 [ 50/188] loss=0.7014 acc=57.0% std=0.2066 LR=1.21e-04
  Ep2 [100/188] loss=0.6123 acc=66.4% std=0.1825 LR=1.61e-04
  Ep2 [150/188] loss=0.5718 acc=69.5% std=0.2056 LR=2.01e-04
[Ep   2/30] tr=66.7% val=70.9% AUC=0.792 AUPR=0.792 F1=0.735 MCC=0.426 Prec=0.675 Rec=0.805 (4.3min)
  ★ Best AUC=0.7925 saved.
  Ep3 [ 50/188] loss=0.5945 acc=67.2% std=0.1993 LR=2.60e-04
  Ep3 [100/188] loss=0.5335 acc=71.9% std=0.2478 LR=2.83e-04
  Ep3 [150/188] loss=0.5543 acc=70.3% std=0.2287 LR=2.97e-04
[Ep   3/30] tr=71.9% val=73.5% AUC=0.823 AUPR=0.822 F1=0.756 MCC=0.477 Prec=0.700 Rec=0.822 (6.4min)
  ★ Best AUC=0.8230 saved.
  Ep4 [ 50/188] loss=0.5284 acc=76.6

In [8]:
# ================================================================
# Cell 8: Test evaluation
# ================================================================
model.load_state_dict(torch.load(CKPT, map_location=device, weights_only=False))
model.eval()

_, te_acc, te_preds, te_labels, te_probs = run_epoch(test_loader, MAX_EPOCH, train=False)

try:
    te_auc  = roc_auc_score(te_labels, te_probs)
    te_aupr = average_precision_score(te_labels, te_probs)
except Exception:
    te_auc = te_aupr = float('nan')

te_f1   = f1_score(te_labels, te_preds, zero_division=0)
te_mcc  = matthews_corrcoef(te_labels, te_preds)
te_prec = precision_score(te_labels, te_preds, zero_division=0)
te_rec  = recall_score(te_labels, te_preds, zero_division=0)
cm      = confusion_matrix(te_labels, te_preds)

print('=' * 55)
print('TEST RESULTS  (KIBA)')
print('=' * 55)
print(f'Acc:  {100*te_acc:.2f}%')
print(f'AUC:  {te_auc:.4f}')
print(f'AUPR: {te_aupr:.4f}')
print(f'F1:   {te_f1:.4f}')
print(f'MCC:  {te_mcc:.4f}')
print(f'Prec: {te_prec:.4f}')
print(f'Rec:  {te_rec:.4f}')
print(f'Confusion: TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}')
print('=' * 55)

TEST RESULTS  (KIBA)
Acc:  81.90%
AUC:  0.9015
AUPR: 0.8928
F1:   0.8212
MCC:  0.6382
Prec: 0.8113
Rec:  0.8313
Confusion: TN=1210 FP=290 FN=253 TP=1247
